# 📡 TelecomX — Predicción de Evasión de Clientes (Churn) · Parte 2

**Programa:** Oracle Next Education (ONE) · Alura Latam  
**Challenge 2 — Data Science · Parte 2**  
**Autor:** Bernardo Adolfo Gómez Montoya

---

> Este notebook es la **continuación directa** de la Parte 1, donde se realizó  
> el pipeline ETL completo sobre los datos de **7.043 clientes** de Telecom X.  
> Aquí aplicamos estadística, análisis de correlación y modelos de  
> **Machine Learning** para predecir qué clientes tienen mayor riesgo de  
> abandonar el servicio.

---

> 🚧 **Notebook en desarrollo** — Las celdas de código se completan  
> progresivamente siguiendo las tarjetas del tablero Trello del challenge.

---
# 🛠️ FASE 1 — Preparación de los Datos
---

## 📌 Tarjeta 1: Extracción del Archivo Tratado

Cargamos el archivo CSV procesado en la **Parte 1** del challenge,  
que ya contiene los datos limpios, estandarizados y traducidos al español.  
Este archivo es el punto de partida directo para el modelado predictivo.

**Archivo de entrada:** `telecomx_limpio.csv` — generado por el notebook de la Parte 1.

In [ ]:
# Tarjeta 1 — Cargar telecomx_limpio.csv y mostrar vista previa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Configuración global de gráficos ──────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Carga del archivo tratado de la Parte 1 ───────────────────────────────────
# Si estás en Google Colab, sube el archivo telecomx_limpio.csv antes de ejecutar
# o cárgalo desde tu Drive:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('telecomx_limpio.csv')

print("=" * 60)
print("📂 ARCHIVO CARGADO EXITOSAMENTE")
print("=" * 60)
print(f"📊 Registros: {df.shape[0]:,}")
print(f"📋 Columnas:  {df.shape[1]}")
print("\n📋 Vista previa:")
display(df.head())

## 📌 Tarjeta 2: Eliminación de Columnas Irrelevantes

Eliminamos columnas que no aportan valor predictivo:
- **`ID_Cliente`** — identificador único, sin capacidad predictiva
- **`Evasion`** — versión texto (conservamos `Evasion_Binaria`)
- **`Cuentas_Diarias`** — derivada directa de `Cargos_Mensuales`, introduce multicolinealidad

También se eliminan los **224 registros con `Evasion_Binaria = -1`** (estado indefinido).

In [ ]:
# Tarjeta 2 — Eliminar columnas irrelevantes y filas con Evasion indefinida
# ── Revisión antes de eliminar ────────────────────────────────────────────────
print("=" * 60)
print("🔍 ESTADO INICIAL DEL DATASET")
print("=" * 60)
print(f"Registros totales:         {df.shape[0]:,}")
print(f"Valores -1 en Evasion_Binaria: {(df['Evasion_Binaria'] == -1).sum()}")
print(f"Nulos en Evasion:          {df['Evasion'].isnull().sum()}")

# ── Eliminar registros con Evasion indefinida ──────────────────────────────────
df = df[df['Evasion_Binaria'].isin([0.0, 1.0])].copy()
df['Evasion_Binaria'] = df['Evasion_Binaria'].astype(int)

# ── Eliminar columnas irrelevantes ────────────────────────────────────────────
cols_eliminar = ['ID_Cliente', 'Evasion', 'Cuentas_Diarias']
df.drop(columns=cols_eliminar, inplace=True)

print("\n" + "=" * 60)
print("✅ DATASET LISTO PARA MODELADO")
print("=" * 60)
print(f"Registros limpios: {df.shape[0]:,}")
print(f"Columnas restantes: {df.shape[1]}")
print(f"\n📋 Columnas actuales:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col} ({df[col].dtype})")

## 📌 Tarjeta 3: Encoding — Codificación de Variables Categóricas

Los algoritmos de Machine Learning requieren entradas numéricas. Aplicamos **One-Hot Encoding** (`pd.get_dummies`) a las variables categóricas con más de 2 categorías, y **Label Encoding binario** (0/1) a las variables con respuesta Sí/No.

**Variables binarias (Yes/No → 1/0):** `Tiene_Pareja`, `Tiene_Dependientes`, `Servicio_Telefono`, `Factura_Digital`

**Variables con múltiples categorías (One-Hot):** `Genero`, `Multiples_Lineas`, `Tipo_Internet`, `Seguridad_Online`, `Backup_Online`, `Proteccion_Dispositivo`, `Soporte_Tecnico`, `Streaming_TV`, `Streaming_Peliculas`, `Tipo_Contrato`, `Metodo_Pago`

In [ ]:
# Tarjeta 3 — Aplicar Label Encoding y One-Hot Encoding
# ── Label Encoding binario (Yes=1 / No=0) ────────────────────────────────────
cols_binarias = [
    'Tiene_Pareja', 'Tiene_Dependientes',
    'Servicio_Telefono', 'Factura_Digital'
]
for col in cols_binarias:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# ── One-Hot Encoding ──────────────────────────────────────────────────────────
cols_ohe = [
    'Genero', 'Multiples_Lineas', 'Tipo_Internet',
    'Seguridad_Online', 'Backup_Online', 'Proteccion_Dispositivo',
    'Soporte_Tecnico', 'Streaming_TV', 'Streaming_Peliculas',
    'Tipo_Contrato', 'Metodo_Pago'
]
df = pd.get_dummies(df, columns=cols_ohe, drop_first=True, dtype=int)

print("=" * 60)
print("✅ ENCODING COMPLETADO")
print("=" * 60)
print(f"Columnas totales tras encoding: {df.shape[1]}")
print(f"Registros:                       {df.shape[0]:,}")
print("\n📋 Columnas generadas:")
for col in df.columns:
    print(f"  • {col}")

## 📌 Tarjeta 4: Verificación de la Proporción de Evasión (Churn)

Antes de modelar, evaluamos si existe **desbalance de clases** en la variable objetivo `Evasion_Binaria`.  
Un fuerte desbalance puede sesgar los modelos hacia la clase mayoritaria, reduciendo su capacidad de detectar correctamente a los clientes que sí evaden.

In [ ]:
# Tarjeta 4 — Calcular proporción de evasión y graficar distribución
# ── Proporción de evasión ─────────────────────────────────────────────────────
conteo = df['Evasion_Binaria'].value_counts()
pct    = df['Evasion_Binaria'].value_counts(normalize=True) * 100

resumen = pd.DataFrame({
    'Categoría': ['No evadió (0)', 'Evadió (1)'],
    'Cantidad':  [conteo[0], conteo[1]],
    'Porcentaje (%)': [pct[0].round(2), pct[1].round(2)]
})
print("=" * 60)
print("📊 PROPORCIÓN DE LA VARIABLE OBJETIVO")
print("=" * 60)
display(resumen)

ratio = conteo[0] / conteo[1]
print(f"\n📌 Ratio desbalanceo: {ratio:.2f}:1  (No evadió : Evadió)")
print("\n💡 El dataset presenta un desbalanceo moderado (~3:1).")
print("   Los modelos basados en árboles (Random Forest) manejan esto")
print("   bien de forma nativa con el parámetro class_weight='balanced'.")
print("   La Regresión Logística también puede ajustarse con este parámetro.")

# ── Gráfico ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribución de la Variable Objetivo — Evasión', fontsize=14, fontweight='bold')

colores = ['#4ECDC4', '#FF6B6B']

axes[0].pie(conteo.values, labels=['No evadió', 'Evadió'],
            autopct='%1.1f%%', colors=colores, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Proporción')

bars = axes[1].bar(['No evadió', 'Evadió'], conteo.values, color=colores, edgecolor='white')
axes[1].set_title('Conteo')
axes[1].set_ylabel('Número de Clientes')
for bar, val in zip(bars, conteo.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 📌 Tarjeta 5: Normalización y Estandarización

Aplicamos **StandardScaler** (media=0, desviación=1) únicamente a las **variables numéricas continuas**: `Meses_Contrato`, `Cargos_Mensuales` y `Cargos_Totales`.

**¿Por qué?**
- La **Regresión Logística** es sensible a la escala: sin normalización, variables con rangos grandes dominan los coeficientes.
- El **Random Forest** no necesita normalización, pero al trabajar con el mismo dataset escalado se garantiza comparabilidad.

Las variables binarias (0/1) generadas por el encoding **no se escalan** — ya están en el mismo rango.

In [ ]:
# Tarjeta 5 — Estandarizar variables numéricas con StandardScaler
from sklearn.preprocessing import StandardScaler

# ── Separar features y target ANTES de escalar ────────────────────────────────
X = df.drop(columns=['Evasion_Binaria'])
y = df['Evasion_Binaria']

# ── Escalar solo variables numéricas continuas ────────────────────────────────
cols_escalar = ['Meses_Contrato', 'Cargos_Mensuales', 'Cargos_Totales']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[cols_escalar] = scaler.fit_transform(X[cols_escalar])

print("=" * 60)
print("✅ ESTANDARIZACIÓN COMPLETADA")
print("=" * 60)
print(f"Variables estandarizadas: {cols_escalar}")
print("\nEstadísticas tras estandarización:")
print(X_scaled[cols_escalar].describe().round(4))
print("\n✅ Variables binarias (0/1): sin cambios — no requieren escala.")

---
# 🎯 FASE 2 — Correlación y Selección de Variables
---

## 📌 Tarjeta 6: Análisis de Correlación

Visualizamos la **matriz de correlación** entre las variables numéricas y la variable objetivo `Evasion_Binaria`.  
Esto nos permite identificar qué variables tienen mayor relación lineal con la evasión y detectar posible **multicolinealidad** entre predictores.

In [ ]:
# Tarjeta 6 — Matriz de correlación y ranking de variables vs Evasion_Binaria
# ── Correlación de variables numéricas con Evasion_Binaria ───────────────────
cols_corr = ['Meses_Contrato', 'Cargos_Mensuales', 'Cargos_Totales',
             'Adulto_Mayor', 'Tiene_Pareja', 'Tiene_Dependientes',
             'Servicio_Telefono', 'Factura_Digital', 'Evasion_Binaria']

corr_matrix = df[cols_corr].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Análisis de Correlación', fontsize=15, fontweight='bold')

# Heatmap completo
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Correlación'})
axes[0].set_title('Matriz de Correlación — Variables Numéricas', fontsize=12)
axes[0].tick_params(axis='x', rotation=35)

# Ranking de correlación con Evasion_Binaria
corr_target = (corr_matrix['Evasion_Binaria']
               .drop('Evasion_Binaria')
               .sort_values(key=abs, ascending=True))
colors_bar = ['#FF6B6B' if v > 0 else '#4ECDC4' for v in corr_target.values]
axes[1].barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlación con Evasión (mayor → menor)', fontsize=12)
axes[1].set_xlabel('Coeficiente de Correlación')
for i, (val, name) in enumerate(zip(corr_target.values, corr_target.index)):
    axes[1].text(val + (0.005 if val >= 0 else -0.005), i,
                 f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\n📊 Ranking de correlación con Evasion_Binaria:")
print(corr_matrix['Evasion_Binaria'].drop('Evasion_Binaria')
      .sort_values(key=abs, ascending=False).round(3).to_string())



## 📌 Tarjeta 7: Análisis Dirigido

Profundizamos en las relaciones más relevantes detectadas en la correlación:  
- **Meses de Contrato vs Evasión**: ¿los clientes nuevos evaden más?  
- **Cargos Totales vs Evasión**: ¿el gasto acumulado distingue a los que se quedan?  
- **Cargos Mensuales vs Evasión**: ¿los planes caros generan más churn?

In [ ]:
# Tarjeta 7 — Histogramas y boxplots de variables numéricas vs Evasión
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Análisis Dirigido — Variables Numéricas vs Evasión', fontsize=14, fontweight='bold')

paleta = {0: '#4ECDC4', 1: '#FF6B6B'}
labels = {0: 'No evadió', 1: 'Evadió'}

pares = [
    ('Meses_Contrato',   'Meses de Contrato (Tenure)'),
    ('Cargos_Mensuales', 'Cargos Mensuales (USD)'),
    ('Cargos_Totales',   'Cargos Totales (USD)'),
]

for ax, (col, titulo) in zip(axes, pares):
    for val in [0, 1]:
        subset = df[df['Evasion_Binaria'] == val][col]
        ax.hist(subset, bins=30, alpha=0.6,
                color=paleta[val], label=labels[val], edgecolor='none')
    ax.set_title(f'{titulo}\nvs Evasión', fontsize=11)
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.tight_layout()
plt.show()

# Boxplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Boxplots — Variables Numéricas por Evasión', fontsize=14, fontweight='bold')

for ax, (col, titulo) in zip(axes, pares):
    sns.boxplot(x='Evasion_Binaria', y=col, data=df,
                palette=paleta, ax=ax)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel('Evasión (0=No, 1=Sí)')
    ax.set_ylabel(titulo)
    medias = df.groupby('Evasion_Binaria')[col].mean()
    for i, (k, v) in enumerate(medias.items()):
        ax.text(i, v, f'  μ={v:.1f}', va='center', fontsize=9, color='navy')

plt.tight_layout()
plt.show()

print("\n📊 Medias por grupo de Evasión:")
print(df.groupby('Evasion_Binaria')[['Meses_Contrato','Cargos_Mensuales','Cargos_Totales']].mean().round(2))

---
# 🤖 FASE 3 — Modelado Predictivo
---

## 📌 Tarjeta 8: Separación de Datos

Dividimos el dataset en **entrenamiento (80%)** y **prueba (20%)**, con `stratify=y` para garantizar que la proporción de evasión sea la misma en ambos subconjuntos.

| Conjunto | Proporción | Uso |
|---|---|---|
| Entrenamiento | 80% | Ajuste de los modelos |
| Prueba | 20% | Evaluación de rendimiento real |

Preparamos dos versiones: `X_train`/`X_test` con datos escalados (para Regresión Logística) y sin escalar (para Random Forest).

In [ ]:
# Tarjeta 8 — Separar datos en entrenamiento y prueba (80/20, stratify=y)
from sklearn.model_selection import train_test_split

# ── Separación con datos originales (para Random Forest) ──────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Separación con datos escalados (para Regresión Logística) ─────────────────
X_train_sc, X_test_sc, _, _ = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

print("=" * 60)
print("✅ DATOS SEPARADOS CORRECTAMENTE")
print("=" * 60)
print(f"Entrenamiento: {X_train.shape[0]:,} registros ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Prueba:        {X_test.shape[0]:,} registros ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nProporción de evasión en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporción de evasión en prueba:        {y_test.mean()*100:.1f}%")
print("✅ Distribución de clases preservada (stratify=y)")

## 📌 Tarjeta 9: Creación y Entrenamiento de Modelos

Entrenamos **dos modelos** con enfoques distintos:

| Modelo | Normalización | Justificación |
|---|---|---|
| **Regresión Logística** | ✅ Sí — datos escalados | Sensible a la escala; optimiza coeficientes mediante gradiente |
| **Random Forest** | ❌ No — datos originales | Basado en árboles de decisión; independiente de la escala |

Ambos modelos usan `class_weight='balanced'` para compensar el desbalanceo moderado de clases (73.5% vs 26.5%).

In [ ]:
# Tarjeta 9 — Entrenar Regresión Logística y Random Forest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ── Modelo 1: Regresión Logística ─────────────────────────────────────────────
print("🔄 Entrenando Regresión Logística...")
modelo_lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    solver='lbfgs'
)
modelo_lr.fit(X_train_sc, y_train)
print("✅ Regresión Logística entrenada.")

# ── Modelo 2: Random Forest ───────────────────────────────────────────────────
print("\n🔄 Entrenando Random Forest...")
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train, y_train)
print("✅ Random Forest entrenado.")
print(f"   Árboles: {modelo_rf.n_estimators}")
print(f"   Profundidad máxima: {modelo_rf.max_depth}")

## 📌 Tarjeta 10: Evaluación de los Modelos

Evaluamos ambos modelos con las métricas estándar de clasificación binaria:

| Métrica | Qué mide |
|---|---|
| **Exactitud (Accuracy)** | % de predicciones correctas totales |
| **Precisión** | De los que predijo como evasión, ¿cuántos realmente evadieron? |
| **Recall** | De los que realmente evadieron, ¿cuántos detectó el modelo? |
| **F1-Score** | Media armónica de Precisión y Recall — balance entre ambas |
| **Matriz de Confusión** | Desglose completo: VP, FP, VN, FN |

> 💡 En un problema de churn, el **Recall** es especialmente importante: queremos minimizar los falsos negativos (clientes que sí van a evadir pero el modelo no detecta).

In [ ]:
# Tarjeta 10 — Evaluar modelos con métricas y matrices de confusión
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score, f1_score)

def evaluar_modelo(nombre, modelo, X_t, y_t):
    y_pred = modelo.predict(X_t)
    acc  = accuracy_score(y_t, y_pred)
    prec = precision_score(y_t, y_pred)
    rec  = recall_score(y_t, y_pred)
    f1   = f1_score(y_t, y_pred)

    print("=" * 60)
    print(f"📊 MODELO: {nombre}")
    print("=" * 60)
    print(f"  Exactitud (Accuracy): {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precisión:            {prec:.4f}")
    print(f"  Recall:               {rec:.4f}")
    print(f"  F1-Score:             {f1:.4f}")
    print("\n  Reporte completo:")
    print(classification_report(y_t, y_pred,
                                target_names=['No evadió (0)', 'Evadió (1)']))
    return y_pred, acc, prec, rec, f1

pred_lr, acc_lr, prec_lr, rec_lr, f1_lr = evaluar_modelo(
    "Regresión Logística", modelo_lr, X_test_sc, y_test)

pred_rf, acc_rf, prec_rf, rec_rf, f1_rf = evaluar_modelo(
    "Random Forest",       modelo_rf, X_test,    y_test)

In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Matrices de Confusión', fontsize=14, fontweight='bold')

modelos_preds = [
    ("Regresión Logística", pred_lr),
    ("Random Forest",       pred_rf),
]

for ax, (nombre, pred) in zip(axes, modelos_preds):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No evadió', 'Evadió'],
                yticklabels=['No evadió', 'Evadió'])
    ax.set_title(f'{nombre}', fontsize=12)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    # Anotaciones adicionales
    vp = cm[1,1]; fn = cm[1,0]; fp = cm[0,1]; vn = cm[0,0]
    ax.set_xlabel(f'Predicho\nVP={vp} | FN={fn} | FP={fp} | VN={vn}')

plt.tight_layout()
plt.show()

# ── Comparativa final ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 COMPARATIVA FINAL DE MODELOS")
print("=" * 60)
comparativa = pd.DataFrame({
    'Modelo':    ['Regresión Logística', 'Random Forest'],
    'Accuracy':  [f'{acc_lr:.4f}', f'{acc_rf:.4f}'],
    'Precisión': [f'{prec_lr:.4f}', f'{prec_rf:.4f}'],
    'Recall':    [f'{rec_lr:.4f}', f'{rec_rf:.4f}'],
    'F1-Score':  [f'{f1_lr:.4f}', f'{f1_rf:.4f}'],
})
display(comparativa)

---
# 📋 FASE 4 — Interpretación y Conclusiones
---

## 📌 Tarjeta 11: Análisis de Importancia de Variables

Analizamos qué variables son más determinantes en cada modelo:

- **Regresión Logística** — coeficientes de cada variable
- **Random Forest** — importancia por reducción de impureza (Gini)

In [11]:
# TODO: Tarjeta 11 — Graficar importancia de variables en ambos modelos

---
# 📝 INFORME FINAL
## TelecomX — Predicción de Evasión de Clientes · Parte 2
---

## 🔹 Introducción

> 🚧 *Se completará al finalizar todas las tarjetas.*

---

## 🔹 Preparación de los Datos

> 🚧 *Se completará al finalizar la Fase 1.*

---

## 🔹 Hallazgos del Análisis de Correlación

> 🚧 *Se completará al finalizar la Fase 2.*

---

## 🔹 Rendimiento de los Modelos

> 🚧 *Se completará al finalizar la Fase 3.*

---

## 🔹 Variables más Importantes

> 🚧 *Se completará al finalizar la Tarjeta 11.*

---

## 🔹 Recomendaciones Estratégicas

> 🚧 *Se completará al finalizar la Fase 4.*

---

*Desarrollado con ❤️ por **Bernardo Adolfo Gómez Montoya** · Alura Latam + Oracle Next Education*

In [12]:
# TODO: Tarjeta 12 — Exportar dataset final preprocesado a CSV